In [ ]:
from pathlib import Path
import csv

# Paths
cv_folds_dir = Path("actreg/dataprep/cv_folds")
clips_root = Path("/orcd/scratch/bcs/001/sensein/sails/rmm/vjepa2_finetune_clips")

cv_folds_dir, clips_root


In [ ]:
rows_vs_clips = []
for csv_path in sorted(cv_folds_dir.glob("*.csv")):
    # Count CSV rows (skip header)
    with csv_path.open("r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        row_count = sum(1 for _ in reader)

    target_dir = clips_root / csv_path.stem
    if not target_dir.exists():
        status = "MISSING_DIR"
        clip_count = 0
    else:
        clip_count = sum(1 for _ in target_dir.glob("*.mp4"))
        status = "OK" if clip_count == row_count else "MISMATCH"

    rows_vs_clips.append((csv_path.name, row_count, clip_count, status, target_dir))
    print(f"{csv_path.name}: rows={row_count} clips={clip_count} status={status} dir={target_dir}")

mismatches = [r for r in rows_vs_clips if r[3] != "OK"]
print("\nMismatches or missing:")
for name, rows, clips, status, tdir in mismatches:
    print(f"- {name}: rows={rows}, clips={clips}, status={status}, dir={tdir}")
if not mismatches:
    print("None")
